# 01 Quality control and rainfall

The processing chain of the released CML files:

1. `scripts/qc_from_raw.py`: quality control of the raw TX and RX (RAW file -> QC file);
2. `scripts/rainfall_1min.py`: Graf RSTD wet/dry detection, constant baseline, Pastorek wet antenna correction and
   ITU-R P.838-3 (2005) rain rate (QC file -> 1-minute rainfall file);
3. `scripts/derive_5min.py`: 5-minute rainfall from the 1-minute rainfall.

The full runs take about 30 min (QC) and 8 min (rainfall) with 12 threads. Here the chain runs for a few CMLs and the
result is compared with the released files. Needs `numba` and `pycomlink`.

In [ ]:
from pathlib import Path
import json, subprocess, sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from essd_cz.config import load_config

CONFIG = ROOT / 'config' / 'user_paths.json'   # copy of config/user_paths.example.json with your paths
cfg = load_config(CONFIG)
paths, OUT = cfg['datasets'], cfg['output_dir']


def run(script, *args):
    """Run a script of the repository with the configuration of this notebook."""
    cmd = [sys.executable, str(ROOT / script), '--config', str(CONFIG), *map(str, args)]
    print(' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)

In [ ]:
CMLS = ['45035_45097_2740', '10001_10066_3107', '10706_10757_3347']
work = OUT / 'example_chain'
work.mkdir(parents=True, exist_ok=True)
qc_out = work / 'CZ_CML_PREPROCESSED_1MIN_20210401_20211031_example.nc'
rain_out = work / 'rain_1min_example.nc'
rain5_out = work / 'rain_5min_example.nc'
if not qc_out.exists():
    run('scripts/qc_from_raw.py', '--only-cmls', ','.join(CMLS), '--tag', 'example', '--out-dir', work)
if not rain_out.exists():
    run('scripts/rainfall_1min.py', '--input-nc', qc_out, '--output-nc', rain_out, '--workers', 4)
if not rain5_out.exists():
    run('scripts/derive_5min.py', '--native', rain_out, '--output', rain5_out)

## Compare with the released files
Every variable of the example output must equal the released file for the same CMLs.

In [ ]:
import numpy as np
import xarray as xr


def same(test, released, variables):
    a = xr.open_dataset(test, mask_and_scale=False, decode_times=False)
    b = xr.open_dataset(released, mask_and_scale=False, decode_times=False).sel(cml_id=a['cml_id'].values)
    for v in variables:
        print(f'{v:25s}', 'identical' if np.array_equal(a[v].values, b[v].values, equal_nan=True) else 'DIFFERENT')


same(qc_out, paths['qc'], ['tx', 'rx', 'trsl', 'qc_flag', 'qc_sublink_status'])
same(rain_out, paths['rain'], ['R_sublink_1min', 'R_mean_1min', 'graf_wet_1min', 'graf_valid_1min'])

## Full runs
```
python scripts/qc_from_raw.py        # -> <output_dir>/qc/
python scripts/rainfall_1min.py      # uses the released QC file; -> <output_dir>/rainfall/
python scripts/derive_5min.py        # uses the released 1-minute rainfall; -> <output_dir>/rainfall_5min/
```